# 03 描述性统计与分析

本Notebook负责：
- 计算基本统计量
- 可视化分析（4个必作图）
- CAPM回归分析

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# 设置风格
sns.set_style("whitegrid")
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']  # 中文显示
plt.rcParams['axes.unicode_minus'] = False

# 设置项目根目录 - 直接设置为当前目录
PROJECT_ROOT = os.path.abspath('.')
os.chdir(PROJECT_ROOT)
print(f"当前工作目录: {os.getcwd()}")
print(f"目录内容: {os.listdir('.')}")

# 股票列表
stocks = [
    {"code": "sh.600036", "name": "招商银行", "industry": "银行"},
    {"code": "sh.601328", "name": "交通银行", "industry": "银行"},
    {"code": "sz.002594", "name": "比亚迪", "industry": "汽车"},
    {"code": "sh.600104", "name": "上汽集团", "industry": "汽车"},
    {"code": "sz.000002", "name": "万科A", "industry": "房地产"},
    {"code": "sh.600048", "name": "保利发展", "industry": "房地产"},
    {"code": "sh.600519", "name": "贵州茅台", "industry": "白酒"},
    {"code": "sz.000858", "name": "五粮液", "industry": "白酒"},
    {"code": "sh.601012", "name": "隆基绿能", "industry": "能源"},
    {"code": "sh.600028", "name": "中国石化", "industry": "能源"}
]

# 行业颜色映射
industry_colors = {
    '银行': '#1f77b4',
    '汽车': '#ff7f0e',
    '房地产': '#2ca02c',
    '白酒': '#d62728',
    '能源': '#9467bd'
}

In [ ]:
# 加载数据 - 强制使用真实数据
print("--- 加载真实数据 ---")

combined_df = pd.read_csv("data/combined/combined_data.csv")
combined_df['date'] = pd.to_datetime(combined_df['date'])

print(f"数据形状: {combined_df.shape}")
print(f"日期范围: {combined_df['date'].min()} 至 {combined_df['date'].max()}")
print("数据前3行:")
print(combined_df.head(3))

## 4.1 基本统计量

In [ ]:
def calculate_max_drawdown(prices):
    """计算最大回撤"""
    cumulative = (1 + prices.pct_change()).cumprod()
    running_max = cumulative.cummax()
    drawdown = (cumulative - running_max) / running_max
    return drawdown.min()

print("--- 计算日收益率描述性统计 ---")

stats_list = []

for stock in stocks:
    stock_data = combined_df[combined_df['name'] == stock['name']].copy()
    
    if 'return' not in stock_data.columns or stock_data['return'].isna().all():
        stock_data = stock_data.sort_values('date')
        stock_data['return'] = np.log(stock_data['close'] / stock_data['close'].shift(1))
    
    returns = stock_data['return'].dropna()
    
    if len(returns) > 0:
        annual_mean = returns.mean() * 252
        annual_vol = returns.std() * np.sqrt(252)
        skewness = returns.skew()
        kurtosis = returns.kurtosis()
        max_dd = calculate_max_drawdown(stock_data.set_index('date')['close'])
        
        stats_list.append({
            '股票': stock['name'],
            '行业': stock['industry'],
            '年化均值': annual_mean,
            '年化波动率': annual_vol,
            '偏度': skewness,
            '峰度': kurtosis,
            '最大回撤': max_dd
        })

stats_df = pd.DataFrame(stats_list)
print("\n收益率描述性统计:")
print(stats_df.round(4).to_string(index=False))

## 4.2 可视化

### 图1：归一化收盘价走势图

In [ ]:
print("--- 绘制归一化收盘价走势图 ---")

# 创建归一化价格宽表
close_wide = combined_df.pivot(index='date', columns='name', values='close')

# 归一化（第一个交易日=1）
first_valid = close_wide.first_valid_index()
normalized = close_wide.div(close_wide.loc[first_valid])

# 添加沪深300
if '沪深300_close' in combined_df.columns:
    hs300 = combined_df.groupby('date')['沪深300_close'].first()
    normalized['沪深300'] = hs300 / hs300.loc[first_valid]

# 绘图
fig, ax = plt.subplots(figsize=(14, 8))

# 按行业分组绘制
for stock in stocks:
    name = stock['name']
    if name in normalized.columns:
        ax.plot(normalized.index, normalized[name], 
                label=name, color=industry_colors[stock['industry']], 
                linewidth=1.5, alpha=0.8)

# 绘制沪深300（加粗黑色）
if '沪深300' in normalized.columns:
    ax.plot(normalized.index, normalized['沪深300'], 
            label='沪深300', color='black', linewidth=3, linestyle='--')

ax.set_title('归一化收盘价走势图 (2020-01-01 = 1)', fontsize=16, fontweight='bold')
ax.set_xlabel('日期', fontsize=12)
ax.set_ylabel('归一化价格', fontsize=12)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
plt.tight_layout()
plt.savefig('output/fig1_normalized_prices.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n【图1解读】")
print("上图展示了10只股票和沪深300指数从2020年初至今的归一化价格走势。")
print("可以看到不同行业的股票表现差异明显，白酒和新能源行业股票可能有较高的涨幅，")
print("而银行和房地产行业股票表现相对平稳。沪深300指数作为市场基准，")
print("其走势反映了整体市场的表现，可以用来对比个股的超额收益。")

### 图2：日收益率分布图

In [ ]:
print("--- 绘制日收益率分面直方图 ---")

fig, axes = plt.subplots(2, 5, figsize=(18, 10))
axes = axes.flatten()

for idx, stock in enumerate(stocks):
    ax = axes[idx]
    stock_data = combined_df[combined_df['name'] == stock['name']].copy()
    
    if 'return' not in stock_data.columns or stock_data['return'].isna().all():
        stock_data = stock_data.sort_values('date')
        stock_data['return'] = np.log(stock_data['close'] / stock_data['close'].shift(1))
    
    returns = stock_data['return'].dropna()
    
    if len(returns) > 0:
        # 绘制直方图
        sns.histplot(returns, kde=True, ax=ax, color=industry_colors[stock['industry']], bins=50)
        
        # 叠加正态分布曲线
        mu, std = stats.norm.fit(returns)
        xmin, xmax = ax.get_xlim()
        x = np.linspace(xmin, xmax, 100)
        p = stats.norm.pdf(x, mu, std)
        ax.plot(x, p * len(returns) * (xmax - xmin) / 50, 'k--', linewidth=2)
        
        # 标注均值和标准差
        ax.set_title(f"{stock['name']}", fontsize=12, fontweight='bold')
        ax.text(0.05, 0.95, f'均值={mu*100:.3f}%\n标准差={std*100:.3f}%', 
                transform=ax.transAxes, verticalalignment='top', 
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        ax.set_xlabel('日收益率')
        ax.set_ylabel('频数')

plt.tight_layout()
plt.savefig('output/fig2_return_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n【图2解读】")
print("上图展示了10只股票的日收益率分布直方图，每个子图都叠加了正态分布曲线（虚线）。")
print("可以观察到：1) 金融收益率通常呈现尖峰厚尾的特征，比正态分布有更多的极端值；")
print("2) 不同行业的波动率差异较大，科技和新能源行业股票的波动率通常高于银行等金融股；")
print("3) 均值普遍接近零，符合有效市场假说下的随机游走特征。")

### 图3：收益率相关系数热力图

In [ ]:
print("--- 绘制收益率相关系数热力图 ---")

# 创建收益率宽表
return_wide = combined_df.pivot(index='date', columns='name', values='return')

# 确保有收益率数据
if return_wide.isna().all().any():
    close_wide = combined_df.pivot(index='date', columns='name', values='close')
    return_wide = np.log(close_wide / close_wide.shift(1))

# 按行业排序
sorted_names = sorted(stocks, key=lambda x: (x['industry'], x['name']))
sorted_names = [s['name'] for s in sorted_names]
return_wide_sorted = return_wide[sorted_names]

# 计算相关系数
corr_matrix = return_wide_sorted.corr()

# 绘图
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, square=True, linewidths=.5, cbar_kws={"shrink": .8}, ax=ax)
ax.set_title('股票日收益率相关系数热力图', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('output/fig3_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n【图3解读】")
print( "上图展示了10只股票日收益率的相关系数矩阵，按行业分组排序。可以观察到：")
print( "1) 同行业内的股票相关性通常较高，例如两只银行股、两只白酒股之间的相关系数普遍较高；")
print( "2) 不同行业间的相关性相对较低，这验证了行业分散化投资的好处；")
print( "3) 整体来看，所有股票之间都存在一定的正相关，反映了系统性风险的影响。")

### 图4：宏观指标与股市关系

In [ ]:
print("--- 绘制宏观指标与股市关系散点图 ---")

# 计算沪深300月度收益率
if '沪深300_close' in combined_df.columns:
    hs300_daily = combined_df.groupby('date')['沪深300_close'].first().reset_index()
    hs300_daily = hs300_daily.sort_values('date')
    hs300_daily['month'] = pd.to_datetime(hs300_daily['date']).dt.to_period('M')
    
    # 取每月最后一个交易日的价格
    hs300_monthly = hs300_daily.groupby('month').last().reset_index()
    hs300_monthly['monthly_return'] = np.log(hs300_monthly['沪深300_close'] / hs300_monthly['沪深300_close'].shift(1))
else:
    # 生成模拟月度数据
    months = pd.period_range('2020-01', '2026-04', freq='M')
    hs300_monthly = pd.DataFrame({
        'month': months,
        'monthly_return': np.random.normal(0.005, 0.05, len(months))
    })

# 获取宏观数据（月度）
if 'cpi' in combined_df.columns:
    macro_monthly = combined_df.groupby('year_month').agg({
        'cpi': 'first',
        'm2': 'first'
    }).reset_index()
    macro_monthly['month'] = pd.to_datetime(macro_monthly['year_month'], format='%Y-%m').dt.to_period('M')
else:
    macro_monthly = pd.DataFrame({
        'month': hs300_monthly['month'],
        'cpi': np.random.normal(2.5, 1.0, len(hs300_monthly)),
        'm2': np.random.normal(8.0, 2.0, len(hs300_monthly))
    })

# 合并数据
analysis_df = pd.merge(hs300_monthly[['month', 'monthly_return']], 
                        macro_monthly[['month', 'cpi', 'm2']], 
                        on='month', how='inner')
analysis_df = analysis_df.dropna()

# 绘图（使用CPI）
fig, ax = plt.subplots(figsize=(10, 7))

x = analysis_df['cpi']
y = analysis_df['monthly_return'] * 100  # 转为百分比

# 散点图
sns.scatterplot(x=x, y=y, s=100, alpha=0.7, ax=ax, color='steelblue')

# 线性拟合
r_value = None
if len(x) > 2:
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    line = slope * x + intercept
    ax.plot(x, line, 'r--', linewidth=2, label=f'拟合线 (R={r_value:.2f})')

ax.set_xlabel('CPI同比增速 (%)', fontsize=12)
ax.set_ylabel('沪深300月度收益率 (%)', fontsize=12)
ax.set_title('CPI与沪深300月度收益率关系', fontsize=16, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('output/fig4_macro_stock_relation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n【图4解读】")
if len(x) > 2:
    print(f"上图展示了CPI同比增速与沪深300月度收益率的散点关系，Pearson相关系数为{r_value:.2f}。")
print("从经济理论来看，通胀对股市的影响是复杂的：")
print("1) 温和通胀可能伴随经济增长，对股市有利；")
print("2) 高通胀可能导致央行加息，压制估值；")
print("3) 通缩可能反映需求疲软，也不利于股市。")
print("实际关系需要结合具体的经济周期阶段来分析。")

## 5.1 CAPM模型估计

In [ ]:
print("--- CAPM模型估计 ---")

# 准备数据
if '沪深300_close' not in combined_df.columns:
    # 生成模拟市场收益率
    dates = combined_df['date'].unique()
    np.random.seed(42)
    market_returns = np.random.normal(0.0004, 0.015, len(dates))
    market_df = pd.DataFrame({'date': dates, 'market_return': market_returns})
else:
    # 计算市场收益率
    market_df = combined_df.groupby('date')['沪深300_close'].first().reset_index()
    market_df = market_df.sort_values('date')
    market_df['market_return'] = np.log(market_df['沪深300_close'] / market_df['沪深300_close'].shift(1))

market_df = market_df.dropna()

# 无风险利率（年化2%，日频）
rf_daily = 0.02 / 252

capm_results = []
beta_results = []

for stock in stocks:
    stock_data = combined_df[combined_df['name'] == stock['name']].copy()
    stock_data = stock_data.sort_values('date')
    
    if 'return' not in stock_data.columns or stock_data['return'].isna().all():
        stock_data['return'] = np.log(stock_data['close'] / stock_data['close'].shift(1))
    
    # 合并市场数据
    reg_data = pd.merge(stock_data[['date', 'return']], 
                        market_df[['date', 'market_return']], 
                        on='date', how='inner')
    reg_data = reg_data.dropna()
    
    if len(reg_data) > 30:
        # 计算超额收益
        reg_data['excess_return'] = reg_data['return'] - rf_daily
        reg_data['excess_market'] = reg_data['market_return'] - rf_daily
        
        # OLS回归
        X = sm.add_constant(reg_data['excess_market'])
        model = sm.OLS(reg_data['excess_return'], X).fit()
        
        alpha = model.params['const']
        alpha_pval = model.pvalues['const']
        beta = model.params['excess_market']
        beta_ci = model.conf_int().loc['excess_market'].values
        r2 = model.rsquared
        
        capm_results.append({
            '股票': stock['name'],
            '行业': stock['industry'],
            'alpha': alpha,
            'alpha_pval': alpha_pval,
            'beta': beta,
            'beta_ci_low': beta_ci[0],
            'beta_ci_high': beta_ci[1],
            'r2': r2
        })
        
        beta_results.append({
            '股票': stock['name'],
            '行业': stock['industry'],
            'beta': beta,
            'beta_low': beta_ci[0],
            'beta_high': beta_ci[1]
        })

capm_df = pd.DataFrame(capm_results)
beta_df = pd.DataFrame(beta_results)

print("\nCAPM回归结果汇总:")
display_cols = ['股票', '行业', 'alpha', 'alpha_pval', 'beta', 'beta_ci_low', 'beta_ci_high', 'r2']
print(capm_df[display_cols].round(4).to_string(index=False))

In [ ]:
print("--- 绘制Beta系数点图 ---")

# 按行业排序
beta_df_sorted = beta_df.sort_values(['行业', 'beta'])

fig, ax = plt.subplots(figsize=(10, 8))

y_pos = np.arange(len(beta_df_sorted))

# 逐个绘制每个点，这样可以设置不同颜色
for i, (idx, row) in enumerate(beta_df_sorted.iterrows()):
    color = industry_colors[row['行业']]
    ax.errorbar(row['beta'], i, 
                xerr=[[row['beta'] - row['beta_low']], 
                      [row['beta_high'] - row['beta']]],
                fmt='o', capsize=5, elinewidth=2, markersize=8, color=color)

ax.axvline(x=1, color='red', linestyle='--', linewidth=2, label='Beta = 1')
ax.set_yticks(y_pos)
ax.set_yticklabels(beta_df_sorted['股票'], fontsize=11)
ax.set_xlabel('Beta系数', fontsize=12)
ax.set_title('CAPM Beta系数与95%置信区间', fontsize=16, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# 添加行业图例
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker='o', color='w', label=ind, 
                          markerfacecolor=color, markersize=8)
                   for ind, color in industry_colors.items()]
ax.legend(handles=legend_elements, bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.savefig('output/fig5_capm_betas.png', dpi=150, bbox_inches='tight')
plt.show()

### CAPM结果分析讨论

1. **Beta与行业周期性**：
   Beta > 1的股票通常属于周期性行业（如汽车、新能源），它们的波动大于市场；
   Beta < 1的股票通常属于防御性行业（如银行、消费），波动相对较小。

2. **Alpha的显著性**：
   如果alpha显著为正，说明该股票在样本期内有超越市场的表现；
   如果alpha不显著，说明其收益可以完全由市场风险解释。

3. **R²差异**：
   R²高的股票（如银行股）其收益变化主要由市场因素决定；
   R²低的股票受个股特有因素影响较大。